In [1]:
%matplotlib inline

# seventh_attempt.ipynb -- Diabetic Retinopathy Detection

**Changes vs sixth_attempt.ipynb (J-series):**
- **(J-1) SE-CustomNetV2**: SEBlock (Squeeze-and-Excitation) inserted after Conv-BN-ReLU in every _block. ~22k new params. No pretrained weights.
- **(J-2) Multi-seed ensemble**: SE-CustomNetV2 trained twice (seeds 42 and 456); test scores averaged for output_custom.csv. Fully valid for CUSTOM category.
- **(J-3) 8-pass TTA**: Full D4 symmetry group (original + hflip + vflip + rot180 + rot90cw + rot90ccw + rot90cw+hflip + rot90ccw+hflip). Both models.
- **(J-4) DenseNet121 Stage 3**: Unfreezes denseblock2 + transition2 with 3-epoch warmup + CosineAnnealingLR(T_max=9).

**CUSTOM validity**: output_custom.csv is produced solely from SE-CustomNetV2 (two seeds averaged). DenseNet121 scores are never blended into the custom submission.


## 1. Imports & Setup

In [2]:
from __future__ import print_function, division
import os, csv
import torch
import pandas as pd
from skimage import io, transform, util, color
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, utils, models
from torchvision.models import densenet121, DenseNet121_Weights
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
import time
import copy
from PIL import Image
from zipfile import ZipFile
import random
import numpy.random as npr
import cv2
import warnings

warnings.filterwarnings('ignore')
random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

plt.ion()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [3]:
DATA_ROOT = '/kaggle/input/datasets/mariamuozperez/lab5-cv/LS5_CV_2025_2026_DB_Retinopathy'

In [4]:
# Run once to extract data, then comment out
# import zipfile
# with zipfile.ZipFile('./db.zip', 'r') as z:
#     z.extractall('./data')

## 2. Dataset

In [5]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, header=0,
                                   dtype={'id': str, 'eye': int, 'label': int})
        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(range(len(self.dataset)))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)
        self.root_dir = root_dir
        self.img_dir  = os.path.join(root_dir, 'images')
        self.transform = transform
        self.levels  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
        self.classes = ['No DR', 'DR']

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = os.path.join(self.img_dir, self.dataset.id[idx] + '.jpg')
        image = io.imread(img_name)
        if self.dataset.eye[idx] == 1:
            image = image[:, ::-1, :]
        sample = {
            'image': image,
            'eye':   self.dataset.eye[idx],
            'label': (self.dataset.label[idx] > 0).astype(dtype=np.int64)
        }
        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Transforms

In [6]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        imgray = color.rgb2gray(image)
        _, mask = cv2.threshold(imgray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return {'image': image, 'eye': eye, 'label': label}
        minx = np.maximum(sidx[1].min() - self.border[1], 0)
        maxx = np.minimum(sidx[1].max() + 1 + self.border[1], w)
        miny = np.maximum(sidx[0].min() - self.border[0], 0)
        maxy = np.minimum(sidx[0].max() + 1 + self.border[1], h)
        image = image[miny:maxy, minx:maxx, ...]
        return {'image': image, 'eye': eye, 'label': label}


class BenGraham(object):
    def __init__(self, sigmaX=10):
        self.sigmaX = sigmaX

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        if image.dtype == np.uint8:
            img_u8 = image
        else:
            img_u8 = (np.clip(image, 0, 1) * 255).astype(np.uint8)
        blurred  = cv2.GaussianBlur(img_u8, (0, 0), self.sigmaX)
        enhanced = cv2.addWeighted(img_u8, 4, blurred, -4, 128)
        enhanced = np.clip(enhanced, 0, 255).astype(np.uint8)
        mask = np.zeros(enhanced.shape, dtype=np.uint8)
        h, w = enhanced.shape[:2]
        cv2.circle(mask, (w // 2, h // 2), int(0.9 * min(h, w) / 2), (1, 1, 1), -1, 8, 0)
        enhanced = enhanced * mask + 128 * (1 - mask)
        return {'image': enhanced.astype(np.float32) / 255.0, 'eye': eye, 'label': label}


class Rescale(object):
    def __init__(self, output_size):
        self.output_size = output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            new_h = self.output_size * h / w if h > w else self.output_size
            new_w = self.output_size if h > w else self.output_size * w / h
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (int(new_h), int(new_w)))
        return {'image': image, 'eye': eye, 'label': label}


class RandomCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = np.random.randint(0, h - new_h) if h > new_h else 0
        left = np.random.randint(0, w - new_w) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class CenterCrop(object):
    def __init__(self, output_size):
        self.output_size = (output_size, output_size) if isinstance(output_size, int) else output_size

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size
        top  = int((h - new_h) / 2) if h > new_h else 0
        left = int((w - new_w) / 2) if w > new_w else 0
        image = image[top:top + new_h, left:left + new_w]
        return {'image': image, 'eye': eye, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        image = torch.from_numpy(image.transpose((2, 0, 1)))
        label = torch.tensor(label, dtype=torch.long)
        return {'image': image, 'eye': eye, 'label': label}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        dtype = image.dtype
        mean = torch.as_tensor(self.mean, dtype=dtype, device=image.device)
        std  = torch.as_tensor(self.std,  dtype=dtype, device=image.device)
        image.sub_(mean[:, None, None]).div_(std[:, None, None])
        return {'image': image, 'eye': eye, 'label': label}


class TVCenterCrop(object):
    def __init__(self, size):
        self.CC = transforms.CenterCrop(size)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.CC(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomHorizontalFlip(object):
    def __init__(self, p=0.5):
        self.flip = transforms.RandomHorizontalFlip(p=p)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.flip(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVRandomRotation(object):
    def __init__(self, degrees=15):
        self.rotate = transforms.RandomRotation(degrees=degrees)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.rotate(pil)))
        return {'image': image, 'eye': eye, 'label': label}


class TVColorJitter(object):
    def __init__(self, brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05):
        self.jitter = transforms.ColorJitter(
            brightness=brightness, contrast=contrast,
            saturation=saturation, hue=hue)

    def __call__(self, sample):
        image, eye, label = sample['image'], sample['eye'], sample['label']
        pil = Image.fromarray(util.img_as_ubyte(image))
        image = util.img_as_float(np.asarray(self.jitter(pil)))
        return {'image': image, 'eye': eye, 'label': label}

## 4. Data Pipelines & DataLoaders

In [7]:
pixel_mean = [0.485, 0.456, 0.406]
pixel_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    TVRandomHorizontalFlip(p=0.5),
    TVRandomRotation(degrees=15),
    TVColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    RandomCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

eval_transform = transforms.Compose([
    CropByEye(0.10, 1),
    BenGraham(sigmaX=10),
    Rescale(256),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=pixel_mean, std=pixel_std),
])

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'train.csv'),
    root_dir=DATA_ROOT, maxSize=0, transform=train_transform)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'val.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_ROOT, 'test.csv'),
    root_dir=DATA_ROOT, transform=eval_transform)
print(f'Train: {len(train_dataset)}  Val: {len(val_dataset)}  Test: {len(test_dataset)}')

Train: 2000  Val: 500  Test: 1000


In [8]:
_sample = train_dataset[0]
_img = _sample['image'].numpy().transpose(1, 2, 0)
print(f'Pipeline output -- dtype: {_img.dtype}, min: {_img.min():.3f}, max: {_img.max():.3f}, mean: {_img.mean():.3f}')
assert abs(_img.mean()) < 0.3, f'BenGraham all-gray bug! mean={_img.mean():.3f}'
print('Sanity check passed.')

Pipeline output -- dtype: float64, min: -1.827, max: 2.152, mean: 0.283
Sanity check passed.


In [9]:
train_labels_bin_for_sampler = (train_dataset.dataset['label'].values > 0).astype(int)
class_counts   = np.bincount(train_labels_bin_for_sampler)
sample_weights = np.where(train_labels_bin_for_sampler == 1,
                          1.0 / class_counts[1], 1.0 / class_counts[0])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_dataset), replacement=True)

train_dataloader = DataLoader(train_dataset, batch_size=64,  sampler=sampler,  num_workers=0)
val_dataloader   = DataLoader(val_dataset,   batch_size=256, shuffle=False, num_workers=0)
test_dataloader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=0)

train_labels_bin = (train_dataset.dataset['label'].values > 0).astype(int)
n_neg = int((train_labels_bin == 0).sum())
n_pos = int((train_labels_bin == 1).sum())
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
print(f'No-DR: {n_neg}  DR: {n_pos}  pos_weight: {pos_weight.item():.3f}')

criterion      = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
image_datasets = {'train': train_dataset, 'val': val_dataset}
dataloaders    = {'train': train_dataloader, 'val': val_dataloader}
dataset_sizes  = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names    = train_dataset.classes

No-DR: 1468  DR: 532  pos_weight: 2.759


## 5. Training & Evaluation Utilities

In [10]:
def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=7, label_smoothing=0.0):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_auc, best_epoch, no_improve = 0.0, -1, 0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            numSamples = dataset_sizes[phase]
            outputs_m  = np.zeros((numSamples,), dtype=float)
            labels_m   = np.zeros((numSamples,), dtype=int)
            running_loss, contSamples = 0.0, 0
            for sample in dataloaders[phase]:
                inputs    = sample['image'].to(device).float()
                labels    = sample['label'].to(device).float()
                batchSize = labels.shape[0]
                optimizer.zero_grad()
                with torch.set_grad_enabled(phase == 'train'):
                    logits = model(inputs).flatten()
                    if label_smoothing > 0.0 and phase == 'train':
                        labels_ls = labels * (1 - label_smoothing) + label_smoothing / 2.0
                        loss = criterion(logits, labels_ls)
                    else:
                        loss = criterion(logits, labels)
                    scores = torch.sigmoid(logits).detach()
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                running_loss += loss.item() * batchSize
                outputs_m[contSamples:contSamples + batchSize] = scores.cpu().numpy()
                labels_m [contSamples:contSamples + batchSize] = labels.cpu().numpy()
                contSamples += batchSize
            if phase == 'train':
                scheduler.step()
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_auc  = metrics.roc_auc_score(labels_m, outputs_m)
            print('{} Loss: {:.4f}  AUC: {:.4f}'.format(phase, epoch_loss, epoch_auc))
            if phase == 'val':
                if epoch_auc > best_auc:
                    best_auc, best_epoch, no_improve = epoch_auc, epoch, 0
                    best_model_wts = copy.deepcopy(model.state_dict())
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        print(f'Early stopping: no improvement for {patience} epochs.')
                        model.load_state_dict(best_model_wts)
                        return model
        print()
    elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(elapsed // 60, elapsed % 60))
    print('Best model: epoch {:d}  val AUC: {:.4f}'.format(best_epoch, best_auc))
    model.load_state_dict(best_model_wts)
    return model

In [11]:
def eval_val_auc(model, name, tta=False):
    model.eval()
    n = len(val_dataset)
    scores_m = np.zeros((n, 1), dtype=float)
    labels_m = np.zeros((n,), dtype=int)
    cont = 0
    with torch.no_grad():
        for sample in val_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                s5 = torch.sigmoid(model(torch.rot90(inputs, 1, [2, 3])))
                s6 = torch.sigmoid(model(torch.rot90(inputs, 3, [2, 3])))
                s7 = torch.sigmoid(model(torch.flip(torch.rot90(inputs, 1, [2, 3]), dims=[3])))
                s8 = torch.sigmoid(model(torch.flip(torch.rot90(inputs, 3, [2, 3]), dims=[3])))
                out = (s1 + s2 + s3 + s4 + s5 + s6 + s7 + s8) / 8.0
            else:
                out = torch.sigmoid(model(inputs))
            scores_m[cont:cont + bs, :] = out.cpu().numpy()
            labels_m[cont:cont + bs]     = sample['label'].numpy()
            cont += bs
    auc    = metrics.roc_auc_score(labels_m, scores_m)
    suffix = ' (TTA-8)' if tta else ''
    print(f'{name}{suffix}  --  val AUC: {auc:.4f}')
    return auc


def test_model(model, tta=False):
    model.eval()
    numSamples  = len(test_dataset)
    outputs_m   = np.zeros((numSamples, 1), dtype=float)
    contSamples = 0
    with torch.no_grad():
        for sample in test_dataloader:
            inputs = sample['image'].to(device).float()
            bs = inputs.shape[0]
            if tta:
                s1 = torch.sigmoid(model(inputs))
                s2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])))
                s3 = torch.sigmoid(model(torch.flip(inputs, dims=[2])))
                s4 = torch.sigmoid(model(torch.flip(inputs, dims=[2, 3])))
                s5 = torch.sigmoid(model(torch.rot90(inputs, 1, [2, 3])))
                s6 = torch.sigmoid(model(torch.rot90(inputs, 3, [2, 3])))
                s7 = torch.sigmoid(model(torch.flip(torch.rot90(inputs, 1, [2, 3]), dims=[3])))
                s8 = torch.sigmoid(model(torch.flip(torch.rot90(inputs, 3, [2, 3]), dims=[3])))
                out = (s1 + s2 + s3 + s4 + s5 + s6 + s7 + s8) / 8.0
            else:
                out = torch.sigmoid(model(inputs))
            outputs_m[contSamples:contSamples + bs, :] = out.cpu().numpy()
            contSamples += bs
    return outputs_m

---
## 6. SE-CustomNetV2 (CUSTOM category)

**(J-1)** SEBlock after Conv-BN-ReLU in every _block. r=8: GAP -> FC(C,C/r) -> ReLU -> FC(C/r,C) -> Sigmoid -> multiply. ~22k new params.

**(J-2)** Trained twice (seeds 42, 456). output_custom.csv = mean test scores. No pretrained weights.


In [12]:
class SEBlock(nn.Module):
    def __init__(self, channels, r=8):
        super().__init__()
        mid = max(channels // r, 4)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc  = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.gap(x).flatten(1)
        w = self.fc(w).view(x.size(0), x.size(1), 1, 1)
        return x * w

In [13]:
class CustomNetV2(nn.Module):
    def __init__(self):
        super().__init__()

        def _block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout),
                nn.ReLU(inplace=True),
                SEBlock(cout),       # J-1: channel attention
                nn.MaxPool2d(2, 2),
            )

        self.features = nn.Sequential(
            _block(3,   32),
            _block(32,  64),
            _block(64,  128),
            _block(128, 256),
            _block(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x).flatten(1)
        return self.classifier(x)

In [14]:
_net = CustomNetV2().to(device)
_inp = next(iter(train_dataloader))['image'].to(device).float()
with torch.no_grad():
    _out = _net(_inp)
print(f'Input: {_inp.shape}  Output: {_out.shape}')
total  = sum(p.numel() for p in _net.parameters())
se_tot = sum(p.numel() for m in _net.modules() if isinstance(m, SEBlock) for p in m.parameters())
print(f'SE-CustomNetV2 -- total params: {total:,}  SE params: {se_tot:,}')
del _net, _inp, _out

Input: torch.Size([64, 3, 224, 224])  Output: torch.Size([64, 1])
SE-CustomNetV2 -- total params: 1,051,229  SE params: 38,972


In [15]:
# J-2: train SE-CustomNetV2 with two seeds; average test scores for output_custom.csv
custom_test_scores_list = []
custom_val_auc_list     = []

for seed in [42, 456]:
    print('\n' + '='*60)
    print(f'Training SE-CustomNetV2  seed={seed}')
    print('='*60)
    random.seed(seed); npr.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    model_c     = CustomNetV2().to(device)
    optimizer_c = optim.AdamW(model_c.parameters(), lr=1e-3, weight_decay=5e-3)
    scheduler_c = lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=50)
    model_c = train_model(model_c, criterion, optimizer_c, scheduler_c,
                          num_epochs=50, patience=10, label_smoothing=0.0)

    torch.save(model_c.state_dict(), f'best_customnetv2_seed{seed}.pth')
    print(f'Saved: best_customnetv2_seed{seed}.pth')

    val_auc = eval_val_auc(model_c, f'SE-CustomNetV2 seed={seed}', tta=True)
    test_sc = test_model(model_c, tta=True)
    custom_val_auc_list.append(val_auc)
    custom_test_scores_list.append(test_sc)

    if seed == 42:
        customNetV2_s42  = model_c
    else:
        customNetV2_s456 = model_c

outputs_custom   = np.mean(custom_test_scores_list, axis=0)
auc_custom_final = float(np.mean(custom_val_auc_list))
print(f'\nSeed 42  val AUC: {custom_val_auc_list[0]:.4f}')
print(f'Seed 456 val AUC: {custom_val_auc_list[1]:.4f}')
print(f'Mean val AUC:     {auc_custom_final:.4f}')


Training SE-CustomNetV2  seed=42
Epoch 0/49
----------
train Loss: 1.1116  AUC: 0.5067
val Loss: 1.1946  AUC: 0.4880

Epoch 1/49
----------
train Loss: 1.0906  AUC: 0.5117
val Loss: 1.1323  AUC: 0.5034

Epoch 2/49
----------
train Loss: 1.0915  AUC: 0.5305
val Loss: 1.3131  AUC: 0.5630

Epoch 3/49
----------
train Loss: 1.0931  AUC: 0.5160
val Loss: 1.1691  AUC: 0.5109

Epoch 4/49
----------
train Loss: 1.0650  AUC: 0.5759
val Loss: 1.1606  AUC: 0.6707

Epoch 5/49
----------
train Loss: 1.0617  AUC: 0.6220
val Loss: 1.2154  AUC: 0.6898

Epoch 6/49
----------
train Loss: 1.0330  AUC: 0.6516
val Loss: 1.2816  AUC: 0.6931

Epoch 7/49
----------
train Loss: 1.0022  AUC: 0.6852
val Loss: 0.9800  AUC: 0.6997

Epoch 8/49
----------
train Loss: 0.9803  AUC: 0.7126
val Loss: 0.9869  AUC: 0.6969

Epoch 9/49
----------
train Loss: 1.0257  AUC: 0.6739
val Loss: 0.9825  AUC: 0.7173

Epoch 10/49
----------
train Loss: 0.9971  AUC: 0.6930
val Loss: 1.0403  AUC: 0.7122

Epoch 11/49
----------
train L

---
## 7. DenseNet121 — Stage 1 (FINE-TUNING category)

Unfreeze denseblock4 + norm5. Head: Dropout(0.5) -> Linear(1024,1).
AdamW lr=3e-4 wd=1e-2, CosineAnnealingLR(T_max=30), 30 epochs, patience=7, label_smoothing=0.05.


In [16]:
ftNet = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
for param in ftNet.parameters():
    param.requires_grad = False
ftNet.classifier = nn.Sequential(nn.Dropout(p=0.5), nn.Linear(1024, 1))
for param in ftNet.features.denseblock4.parameters():
    param.requires_grad = True
for param in ftNet.features.norm5.parameters():
    param.requires_grad = True
ftNet = ftNet.to(device)
trainable = sum(p.numel() for p in ftNet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in ftNet.parameters())
print(f'DenseNet121  Trainable: {trainable:,} / {total:,} params')

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 155MB/s]

DenseNet121  Trainable: 2,161,153 / 6,954,881 params


In [17]:
optimizer_ft_s1 = optim.AdamW(
    filter(lambda p: p.requires_grad, ftNet.parameters()),
    lr=3e-4, weight_decay=1e-2)
scheduler_ft_s1 = lr_scheduler.CosineAnnealingLR(optimizer_ft_s1, T_max=30)

In [18]:
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
ftNet = train_model(ftNet, criterion, optimizer_ft_s1, scheduler_ft_s1,
                    num_epochs=30, patience=7, label_smoothing=0.05)

Epoch 0/29
----------
train Loss: 1.1155  AUC: 0.6067
val Loss: 0.9840  AUC: 0.7096

Epoch 1/29
----------
train Loss: 0.9758  AUC: 0.7370
val Loss: 1.0175  AUC: 0.7619

Epoch 2/29
----------
train Loss: 0.8849  AUC: 0.7983
val Loss: 0.9526  AUC: 0.7573

Epoch 3/29
----------
train Loss: 0.8470  AUC: 0.8297
val Loss: 0.8903  AUC: 0.7499

Epoch 4/29
----------
train Loss: 0.8190  AUC: 0.8445
val Loss: 0.9011  AUC: 0.7659

Epoch 5/29
----------
train Loss: 0.7454  AUC: 0.8818
val Loss: 0.9308  AUC: 0.7214

Epoch 6/29
----------
train Loss: 0.7346  AUC: 0.8847
val Loss: 1.0283  AUC: 0.7321

Epoch 7/29
----------
train Loss: 0.6933  AUC: 0.9044
val Loss: 0.9114  AUC: 0.7662

Epoch 8/29
----------
train Loss: 0.6573  AUC: 0.9182
val Loss: 0.9538  AUC: 0.7401

Epoch 9/29
----------
train Loss: 0.6274  AUC: 0.9284
val Loss: 1.0179  AUC: 0.7306

Epoch 10/29
----------
train Loss: 0.5869  AUC: 0.9443
val Loss: 1.0397  AUC: 0.7321

Epoch 11/29
----------
train Loss: 0.5559  AUC: 0.9506
val Loss:

In [19]:
torch.save(ftNet.state_dict(), 'best_densenet121_s1.pth')
print('Saved: best_densenet121_s1.pth')
auc_ft_s1     = eval_val_auc(ftNet, 'DenseNet121 Stage 1', tta=False)
auc_ft_s1_tta = eval_val_auc(ftNet, 'DenseNet121 Stage 1', tta=True)

Saved: best_densenet121_s1.pth
DenseNet121 Stage 1  --  val AUC: 0.7662
DenseNet121 Stage 1 (TTA-8)  --  val AUC: 0.7904


---
## 8. DenseNet121 — Stage 2: Unfreeze denseblock3 + transition3

3-epoch linear warmup + CosineAnnealingLR(T_max=12). 15 epochs, patience=5.
Reverts to Stage 1 if Stage 2 val AUC is lower.


In [20]:
ftNet.load_state_dict(torch.load('best_densenet121_s1.pth', map_location=device))
for param in ftNet.features.denseblock3.parameters():
    param.requires_grad = True
for param in ftNet.features.transition3.parameters():
    param.requires_grad = True
trainable_s2 = sum(p.numel() for p in ftNet.parameters() if p.requires_grad)
print(f'Stage 2 trainable params: {trainable_s2:,}')

optimizer_ft_s2 = optim.AdamW([
    {'params': ftNet.features.denseblock3.parameters(),  'lr': 3e-5},
    {'params': ftNet.features.transition3.parameters(),  'lr': 3e-5},
    {'params': ftNet.features.denseblock4.parameters(),  'lr': 3e-5},
    {'params': ftNet.features.norm5.parameters(),        'lr': 3e-5},
    {'params': ftNet.classifier.parameters(),            'lr': 1e-4},
], weight_decay=1e-2)
warmup_s2 = lr_scheduler.LinearLR(optimizer_ft_s2, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine_s2 = lr_scheduler.CosineAnnealingLR(optimizer_ft_s2, T_max=12)
scheduler_ft_s2 = lr_scheduler.SequentialLR(optimizer_ft_s2,
                    schedulers=[warmup_s2, cosine_s2], milestones=[3])

Stage 2 trainable params: 5,525,249


In [21]:
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
ftNet_s2 = train_model(ftNet, criterion, optimizer_ft_s2, scheduler_ft_s2,
                       num_epochs=15, patience=5, label_smoothing=0.05)

Epoch 0/14
----------
train Loss: 0.5352  AUC: 0.9600
val Loss: 0.9265  AUC: 0.7643

Epoch 1/14
----------
train Loss: 0.4911  AUC: 0.9741
val Loss: 0.9096  AUC: 0.7626

Epoch 2/14
----------
train Loss: 0.4426  AUC: 0.9858
val Loss: 0.9004  AUC: 0.7611

Epoch 3/14
----------
train Loss: 0.4082  AUC: 0.9909
val Loss: 0.9184  AUC: 0.7588

Epoch 4/14
----------
train Loss: 0.3595  AUC: 0.9961
val Loss: 0.9666  AUC: 0.7483

Epoch 5/14
----------
train Loss: 0.3319  AUC: 0.9968
val Loss: 0.9936  AUC: 0.7407
Early stopping: no improvement for 5 epochs.


In [22]:
auc_ft_s2_tta = eval_val_auc(ftNet_s2, 'DenseNet121 Stage 2', tta=True)

if auc_ft_s2_tta > auc_ft_s1_tta:
    print(f'Stage 2 better ({auc_ft_s2_tta:.4f} > {auc_ft_s1_tta:.4f}). Proceeding to Stage 3.')
    torch.save(ftNet_s2.state_dict(), 'best_densenet121_s2.pth')
    best_ftNet_prev  = ftNet_s2
    best_ft_auc_prev = auc_ft_s2_tta
    best_prev_pth    = 'best_densenet121_s2.pth'
else:
    print(f'Stage 2 did NOT improve ({auc_ft_s2_tta:.4f} <= {auc_ft_s1_tta:.4f}). Reverting to Stage 1.')
    ftNet.load_state_dict(torch.load('best_densenet121_s1.pth', map_location=device))
    best_ftNet_prev  = ftNet
    best_ft_auc_prev = auc_ft_s1_tta
    best_prev_pth    = 'best_densenet121_s1.pth'

DenseNet121 Stage 2 (TTA-8)  --  val AUC: 0.7916
Stage 2 better (0.7916 > 0.7904). Proceeding to Stage 3.


---
## 9. DenseNet121 — Stage 3: Unfreeze denseblock2 + transition2 (J-4)

Newly unfrozen layers use lr=1e-5 (lower than Stage 2's 3e-5 -- further from task-adapted features).
3-epoch linear warmup + CosineAnnealingLR(T_max=9). 12 epochs, patience=4.
Reverts to Stage 2 weights if Stage 3 val AUC is lower.


In [23]:
ftNet.load_state_dict(torch.load(best_prev_pth, map_location=device))
for param in ftNet.features.denseblock2.parameters():
    param.requires_grad = True
for param in ftNet.features.transition2.parameters():
    param.requires_grad = True
trainable_s3 = sum(p.numel() for p in ftNet.parameters() if p.requires_grad)
print(f'Stage 3 trainable params: {trainable_s3:,}')

optimizer_ft_s3 = optim.AdamW([
    {'params': ftNet.features.denseblock2.parameters(),  'lr': 1e-5},  # newly unfrozen
    {'params': ftNet.features.transition2.parameters(),  'lr': 1e-5},
    {'params': ftNet.features.denseblock3.parameters(),  'lr': 3e-5},  # Stage 2 warm
    {'params': ftNet.features.transition3.parameters(),  'lr': 3e-5},
    {'params': ftNet.features.denseblock4.parameters(),  'lr': 3e-5},  # Stage 1 warm
    {'params': ftNet.features.norm5.parameters(),        'lr': 3e-5},
    {'params': ftNet.classifier.parameters(),            'lr': 1e-4},
], weight_decay=1e-2)
warmup_s3 = lr_scheduler.LinearLR(optimizer_ft_s3, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine_s3 = lr_scheduler.CosineAnnealingLR(optimizer_ft_s3, T_max=9)
scheduler_ft_s3 = lr_scheduler.SequentialLR(optimizer_ft_s3,
                    schedulers=[warmup_s3, cosine_s3], milestones=[3])

Stage 3 trainable params: 6,577,025


In [24]:
random.seed(42); npr.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
ftNet_s3 = train_model(ftNet, criterion, optimizer_ft_s3, scheduler_ft_s3,
                       num_epochs=12, patience=4, label_smoothing=0.05)

Epoch 0/11
----------
train Loss: 0.5001  AUC: 0.9708
val Loss: 0.9313  AUC: 0.7630

Epoch 1/11
----------
train Loss: 0.4860  AUC: 0.9751
val Loss: 0.9068  AUC: 0.7613

Epoch 2/11
----------
train Loss: 0.4375  AUC: 0.9866
val Loss: 0.9067  AUC: 0.7583

Epoch 3/11
----------
train Loss: 0.4039  AUC: 0.9913
val Loss: 0.9234  AUC: 0.7577

Epoch 4/11
----------
train Loss: 0.3558  AUC: 0.9963
val Loss: 0.9744  AUC: 0.7450
Early stopping: no improvement for 4 epochs.


In [25]:
auc_ft_s3_tta = eval_val_auc(ftNet_s3, 'DenseNet121 Stage 3', tta=True)

if auc_ft_s3_tta > best_ft_auc_prev:
    print(f'Stage 3 better ({auc_ft_s3_tta:.4f} > {best_ft_auc_prev:.4f}). Using Stage 3.')
    torch.save(ftNet_s3.state_dict(), 'best_densenet121_s3.pth')
    best_ftNet   = ftNet_s3
    auc_ft_final = auc_ft_s3_tta
else:
    print(f'Stage 3 did NOT improve ({auc_ft_s3_tta:.4f} <= {best_ft_auc_prev:.4f}). Reverting.')
    best_ftNet   = best_ftNet_prev
    auc_ft_final = best_ft_auc_prev

print(f'Best ftNet val AUC (TTA-8): {auc_ft_final:.4f}')

DenseNet121 Stage 3 (TTA-8)  --  val AUC: 0.7892
Stage 3 did NOT improve (0.7892 <= 0.7916). Reverting.
Best ftNet val AUC (TTA-8): 0.7916


---
## 10. Generate Test Outputs & Submit

output_custom.csv = mean of SE-CustomNetV2 seed 42 + seed 456 (8-pass TTA). No DenseNet blend.

output_ft.csv = best DenseNet121 stage (8-pass TTA).


In [26]:
print('=== Final validation AUC check ===')
print(f'SE-CustomNetV2 seed=42  : {custom_val_auc_list[0]:.4f} (TTA-8)')
print(f'SE-CustomNetV2 seed=456 : {custom_val_auc_list[1]:.4f} (TTA-8)')
print(f'SE-CustomNetV2 mean     : {auc_custom_final:.4f}')
print(f'Best ftNet              : {auc_ft_final:.4f} (TTA-8)')

=== Final validation AUC check ===
SE-CustomNetV2 seed=42  : 0.7671 (TTA-8)
SE-CustomNetV2 seed=456 : 0.7486 (TTA-8)
SE-CustomNetV2 mean     : 0.7579
Best ftNet              : 0.7916 (TTA-8)


In [27]:
# outputs_custom = mean of seeds 42+456, computed in the multi-seed loop above
outputs_ft_submit = test_model(best_ftNet, tta=True)

assert outputs_custom.shape    == (1000, 1)
assert outputs_ft_submit.shape == (1000, 1)
assert np.isfinite(outputs_custom).all()
assert np.isfinite(outputs_ft_submit).all()
print('Checks passed.')
print(f'Custom score range: [{outputs_custom.min():.4f}, {outputs_custom.max():.4f}]')
print(f'FT     score range: [{outputs_ft_submit.min():.4f}, {outputs_ft_submit.max():.4f}]')

Checks passed.
Custom score range: [0.4444, 1.0000]
FT     score range: [0.0234, 0.9995]


In [28]:
with open('output_custom.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_custom)
with open('output_ft.csv', mode='w', newline='') as f:
    csv.writer(f).writerows(outputs_ft_submit)
print('Written: output_custom.csv  output_ft.csv')

Written: output_custom.csv  output_ft.csv


In [29]:
with ZipFile('./codabench_submission.zip', 'w') as zf:
    zf.write('./output_custom.csv')
    zf.write('./output_ft.csv')
print('Created: codabench_submission.zip')
print(f'\nFinal val AUC summary (seventh_attempt.ipynb):')
print(f'  CUSTOM  (SE-CustomNetV2 2-seed ensemble, TTA-8): {auc_custom_final:.4f}')
print(f'  FT      (DenseNet121 best stage, TTA-8):         {auc_ft_final:.4f}')

Created: codabench_submission.zip

Final val AUC summary (seventh_attempt.ipynb):
  CUSTOM  (SE-CustomNetV2 2-seed ensemble, TTA-8): 0.7579
  FT      (DenseNet121 best stage, TTA-8):         0.7916
